In [ ]:
"""
SHAP-Based Feature Importance Analysis for Hardness Prediction.

This script trains a Gradient Boosting Regressor and evaluates feature importance using SHAP values. Publication-quality summary and bar plots are automatically generated and exported.
"""

import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["font.size"] = 18
plt.rcParams["axes.unicode_minus"] = False


# =============================================================================
# Initialize Output Directory
# =============================================================================

output_dir = r".\data\figure"
os.makedirs(output_dir, exist_ok=True)


# =============================================================================
# Load Dataset and Preprocess Features
# =============================================================================

df_filtered = pd.read_csv(r".\data\c_cleaned_hv_with_features.csv")

target_column = "HV"
drop_cols = ["FILE_NAME", target_column]
feature_cols = [c for c in df_filtered.columns if c not in drop_cols]

df_filtered[feature_cols] = df_filtered[feature_cols].fillna(0)

X = df_filtered[feature_cols].values
y = df_filtered[target_column].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# =============================================================================
# Train Gradient Boosting Regressor
# =============================================================================

print("[INFO] Training Gradient Boosting Regressor...")

gb_model = GradientBoostingRegressor(n_estimators=200, random_state=42)
gb_model.fit(X_scaled, y)


# =============================================================================
# Compute SHAP Values
# =============================================================================

print("[INFO] Computing SHAP values using TreeExplainer...")

explainer = shap.TreeExplainer(gb_model)
shap_values = explainer(X_scaled)
shap_values.feature_names = feature_cols


# =============================================================================
# Extract Top Features
# =============================================================================

importance_indices = np.argsort(np.abs(shap_values.values).mean(0))[::-1]
top_5_indices = importance_indices[:5]
shap_values_top5 = shap_values[:, top_5_indices]

print("[INFO] Generating SHAP visualizations...")


# =============================================================================
# SHAP Summary Plot
# =============================================================================

fig1 = plt.figure(figsize=(8, 3.5))

shap.summary_plot(shap_values_top5, max_display=5, plot_type="dot", show=False)

ax1 = plt.gca()

ax1.tick_params(axis="both", labelsize=12)
ax1.set_xlabel("SHAP Value (Impact on Predicted HV)", fontsize=12, fontweight="bold", labelpad=12)

if len(fig1.axes) > 1:
    fig1.axes[-1].tick_params(labelsize=12)
    fig1.axes[-1].set_ylabel("Feature Value", fontsize=12, fontweight="bold", labelpad=12)

plt.subplots_adjust(left=0.30, right=0.93, top=0.75, bottom=0.32)

summary_save_path = os.path.join(output_dir, "gb_shap_summary_plot_top5.png")

plt.savefig(summary_save_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"[INFO] Saved figure: {summary_save_path}")


# =============================================================================
# SHAP Bar Plot
# =============================================================================

plt.figure(figsize=(15, 5))

ax2 = plt.gca()

top_5_features = [feature_cols[i] for i in top_5_indices]
top_5_values = np.abs(shap_values.values).mean(0)[top_5_indices]

top_5_features = top_5_features[::-1]
top_5_values = top_5_values[::-1]

cmap = plt.cm.get_cmap("viridis")
colors = cmap(top_5_values / max(top_5_values))

bars = ax2.barh(top_5_features, top_5_values, color=colors, edgecolor="#222222", height=0.55)

ax2.tick_params(labelsize=18, left=False)
ax2.set_xlabel("Mean |SHAP Value| (Average Impact)", fontsize=20, fontweight="bold", labelpad=12)

ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
ax2.spines["left"].set_color("#cccccc")
ax2.spines["bottom"].set_color("#cccccc")

ax2.grid(axis="x", linestyle="--", alpha=0.4, color="#888888")
ax2.set_axisbelow(True)

max_val = max(top_5_values)

for bar in bars:
    width = bar.get_width()
    ax2.text(width + (max_val * 0.012), bar.get_y() + bar.get_height() / 2, f"{width:.3f}", va="center", ha="left", fontsize=18, color="#222222", fontweight="bold")

ax2.set_xlim(0, max_val * 1.12)

plt.subplots_adjust(left=0.30, right=0.93, top=0.90, bottom=0.18)

bar_save_path = os.path.join(output_dir, "gb_shap_bar_plot_top5.png")

plt.savefig(bar_save_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"[INFO] Saved figure: {bar_save_path}")


# =============================================================================
# Export Summary
# =============================================================================

print("\n[INFO] SHAP analysis completed successfully.")
print(f"[INFO] Summary plot saved to: {summary_save_path}")
print(f"[INFO] Bar plot saved to: {bar_save_path}")
print(f"[INFO] All figures exported to: {output_dir}")


[INFO] Initiating model benchmarking for 6 regression models...

Model                |       R² |     RMSE |      MAE
-------------------------------------------------------
Ridge                |    0.697 |    1.552 |    1.215
Lasso                |    0.672 |    1.614 |    1.269
SVR                  |    0.785 |    1.307 |    1.031
RandomForest         |    0.843 |    1.116 |    0.831
GradientBoosting     |    0.875 |    0.996 |    0.766
XGBoost              |    0.850 |    1.091 |    0.807

[INFO] Model performance ranking (sorted by R²)
           Model    R2  RMSE   MAE
GradientBoosting 0.875 0.996 0.766
         XGBoost 0.850 1.091 0.807
    RandomForest 0.843 1.116 0.831
             SVR 0.785 1.307 1.031
           Ridge 0.697 1.552 1.215
           Lasso 0.672 1.614 1.269

[INFO] Model benchmarking completed successfully.
[INFO] Performance logs saved to: ./data/d_model_comparison_results.csv
[INFO] Summary figures exported to: ./data/figure
